# 01 ? DILIst Dataset Construction and Provenance

This notebook documents the recorded source-to-cohort lineage and verifies the accepted current modeling file. It is a provenance and integrity check, not a claim to recreate the unavailable historical row membership.

## 1. Reproducibility and provenance statement

The accepted current modeling input is `DILIst_Final.csv` (1,198 rows; SHA-256 `4a3c37c704d02b6d0bb80e0727295ca8ba2fd988c4f68790a07a231b03dc32c7`). Structural modeling excludes one RDKit-invalid structure, DILIST_ID 397, leaving 1,197 valid structures. The historical Notebook 1 filter reported 1,095 rows, while the present reconstruction of that filter yields 1,085. The exact historical membership is unavailable; **the cause of the discrepancy is not established**.

Notebook 2 used six manually supplied SMILES replacements. Their source status remains `manual_source_unverified`. The accepted file is treated as frozen; this notebook only reads it and compares an in-memory reconstruction candidate.

## 2. Source dataset description

The audited source workbook `DILIst.xlsx` contains 1,279 records and four source fields: `DILIST_ID`, `CompoundName`, `DILIst Classification `, and `Routs of Administration `. The recorded completed-SMILES file adds the `SMILES` field and contains 1,279 rows. Target labels are coded 0 and 1; the source audit documents 0 as DILI Negative and 1 as DILI Positive.

The completed-SMILES file is an existing recorded artifact. It is not regenerated here. Its audited SHA-256 is `cd1cc723736b247dac4a972ab839e40095a7243dc49715ca214354ff39942160`.

## 3. Recorded historical SMILES acquisition workflow

Original Notebook 1 recorded this sequence:

1. PubChem lookup using the compound name.
2. If unresolved, NCI Cactus name-to-CAS lookup, followed by PubChem lookup by CAS.
3. If still unresolved, direct NCI Cactus SMILES lookup.

This describes the historical code only. No live service requests are made in this notebook, and the workflow description does not establish the source provenance of individual returned structures or of the six manual replacements.

## 4. Load and verify the recorded files

The following checks read the input CSVs only. They do not write, normalize, or replace either source file.

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors

# Resolve the repository root from the notebook working directory or its parents.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "DILIst_SMILES_Completed.csv").is_file()
             and (p / "DILIst_Final.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError(
        "Could not find both accepted CSV inputs from the notebook working directory."
    )

COMPLETED_PATH = ROOT / "DILIst_SMILES_Completed.csv"
ACCEPTED_PATH = ROOT / "DILIst_Final.csv"
EXPECTED_COMPLETED_SHA256 = "cd1cc723736b247dac4a972ab839e40095a7243dc49715ca214354ff39942160"
EXPECTED_ACCEPTED_SHA256 = "4a3c37c704d02b6d0bb80e0727295ca8ba2fd988c4f68790a07a231b03dc32c7"
TARGET = "DILIst Classification "
EXPECTED_COLUMNS = [
    "DILIST_ID", "CompoundName", TARGET,
    "Routs of Administration ", "SMILES"
]

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

completed_sha256 = sha256_file(COMPLETED_PATH)
accepted_sha256 = sha256_file(ACCEPTED_PATH)
completed = pd.read_csv(COMPLETED_PATH)
accepted = pd.read_csv(ACCEPTED_PATH)

print(f"Completed-SMILES file: {COMPLETED_PATH.name}")
print(f"Completed-SMILES SHA-256: {completed_sha256}")
print(f"RDKit version: {rdBase.rdkitVersion}")
print(f"Accepted file: {ACCEPTED_PATH.name}")
print(f"Accepted SHA-256: {accepted_sha256}")
print(f"Accepted SHA matches expected: {accepted_sha256 == EXPECTED_ACCEPTED_SHA256}")

assert completed_sha256 == EXPECTED_COMPLETED_SHA256, "Completed-SMILES input hash differs from audited artifact."
assert accepted_sha256 == EXPECTED_ACCEPTED_SHA256, "Accepted dataset hash differs from the frozen expected artifact."
assert list(completed.columns) == EXPECTED_COLUMNS
assert list(accepted.columns) == EXPECTED_COLUMNS
assert len(completed) == 1279
assert len(accepted) == 1198
assert TARGET in accepted.columns


Completed-SMILES file: DILIst_SMILES_Completed.csv
Completed-SMILES SHA-256: cd1cc723736b247dac4a972ab839e40095a7243dc49715ca214354ff39942160
RDKit version: 2026.03.1
Accepted file: DILIst_Final.csv
Accepted SHA-256: 4a3c37c704d02b6d0bb80e0727295ca8ba2fd988c4f68790a07a231b03dc32c7
Accepted SHA matches expected: True


In [2]:
def schema_profile(frame, name):
    return pd.DataFrame({
        "dataset": name,
        "column": frame.columns,
        "dtype": [str(frame[c].dtype) for c in frame.columns],
        "missing_count": [int(frame[c].isna().sum()) for c in frame.columns],
    })

display(pd.concat([
    schema_profile(completed, "DILIst_SMILES_Completed.csv"),
    schema_profile(accepted, "DILIst_Final.csv"),
], ignore_index=True))

accepted_class_counts = accepted[TARGET].value_counts(dropna=False).sort_index()
accepted_profile = pd.DataFrame({
    "measure": [
        "rows", "columns", "unique DILIST_ID", "duplicate DILIST_ID rows",
        "missing target", "missing SMILES", "class 0", "class 1"
    ],
    "value": [
        len(accepted), accepted.shape[1], accepted["DILIST_ID"].nunique(),
        int(accepted["DILIST_ID"].duplicated().sum()),
        int(accepted[TARGET].isna().sum()), int(accepted["SMILES"].isna().sum()),
        int(accepted_class_counts.get(0, 0)), int(accepted_class_counts.get(1, 0))
    ]
})
display(accepted_profile)


,dataset,column,dtype,missing_count
0,DILIst_SMILES_Completed.csv,DILIST_ID,int64,0
1,DILIst_SMILES_Completed.csv,CompoundName,str,0
2,DILIst_SMILES_Completed.csv,DILIst Classification,int64,0
3,DILIst_SMILES_Completed.csv,Routs of Administration,str,266
4,DILIst_SMILES_Completed.csv,SMILES,str,87
5,DILIst_Final.csv,DILIST_ID,int64,0
6,DILIst_Final.csv,CompoundName,str,0
7,DILIst_Final.csv,DILIst Classification,int64,0
8,DILIst_Final.csv,Routs of Administration,str,257
9,DILIst_Final.csv,SMILES,str,0


,measure,value
0,rows,1198
1,columns,5
2,unique DILIST_ID,1198
3,duplicate DILIST_ID rows,0
4,missing target,0
5,missing SMILES,0
6,class 0,463
7,class 1,735


## 5. Accepted-cohort structural validity

SMILES are parsed with RDKit for verification only. Invalid records are retained in the accepted CSV; they are identified here and omitted only from structure-based modeling inputs. The invalid SMILES string itself is not repeated in the report.

In [3]:
def parse_smiles(value):
    if pd.isna(value):
        return None
    try:
        return Chem.MolFromSmiles(str(value))
    except Exception:
        return None

accepted_mols = [parse_smiles(value) for value in accepted["SMILES"]]
valid_mask = pd.Series([mol is not None for mol in accepted_mols], index=accepted.index)
invalid_rows = accepted.loc[~valid_mask, ["DILIST_ID", "CompoundName", TARGET]].copy()
valid_class_counts = accepted.loc[valid_mask, TARGET].value_counts().sort_index()

print(f"Accepted input rows: {len(accepted)}")
print(f"Valid structures: {int(valid_mask.sum())}")
print(f"Invalid/missing structures: {int((~valid_mask).sum())}")
print("Invalid structure records:")
display(invalid_rows)
print("Class distribution before structure filtering:")
display(accepted[TARGET].value_counts().sort_index().rename_axis("class").to_frame("count"))
print("Class distribution among valid structures:")
display(valid_class_counts.rename_axis("class").to_frame("count"))

assert len(accepted) == 1198
assert int(valid_mask.sum()) == 1197
assert invalid_rows["DILIST_ID"].tolist() == [397]
assert int(accepted_class_counts.get(0, 0)) == 463
assert int(accepted_class_counts.get(1, 0)) == 735
assert int(valid_class_counts.get(0, 0)) == 462
assert int(valid_class_counts.get(1, 0)) == 735


Accepted input rows: 1198


Valid structures: 1197
Invalid/missing structures: 1
Invalid structure records:


[23:43:37] WARNING: not removing hydrogen atom without neighbors
[23:43:37] SMILES Parse Error: syntax error while parsing: [Fe](|[N]=O)(|[C]#N)(|[C]#N)(|[C]#N)(|[C]#N)|[C]#N
[23:43:37] SMILES Parse Error: check for mistakes around position 6:
[23:43:37] [Fe](|[N]=O)(|[C]#N)(|[C]#N)(|[C]#N)(|[C]
[23:43:37] ~~~~~^
[23:43:37] SMILES Parse Error: Failed parsing SMILES '[Fe](|[N]=O)(|[C]#N)(|[C]#N)(|[C]#N)(|[C]#N)|[C]#N' for input: '[Fe](|[N]=O)(|[C]#N)(|[C]#N)(|[C]#N)(|[C]#N)|[C]#N'
[23:43:37] WARNING: not removing hydrogen atom without neighbors
[23:43:37] WARNING: not removing hydrogen atom without neighbors


,DILIST_ID,CompoundName,DILIst Classification
386,397,nitroprusside,0


Class distribution before structure filtering:


,count
class,
0,463
1,735


Class distribution among valid structures:


,count
class,
0,462
1,735


## 6. Notebook 1 filtering branch (current reconstruction)

The current reconstructed branch applies the recorded Notebook 1 filters in order to `DILIst_SMILES_Completed.csv`: missing SMILES, dot-containing structures, RDKit parse failure, molecular weight greater than or equal to 1,000, absence of carbon, then the original explicit metal-symbol set. This is a reconstruction from the available completed file, not proof of historical row membership.

In [4]:
# This is the explicit metal set from the original Notebook 1 code.
METALS = set("Li Be Na Mg Al K Ca Sc Ti V Cr Mn Fe Co Ni Cu Zn Ga Rb Sr Y Zr Nb Mo Tc Ru Rh Pd Ag Cd In Sn Sb Cs Ba La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Ta W Re Os Ir Pt Au Hg Tl Pb Bi Po".split())

stages = []
def record_stage(name, before, after, removed_ids=None):
    stages.append({
        "stage": name,
        "start_rows": int(before),
        "removed": int(before - after),
        "end_rows": int(after),
        "removed_ids": [] if removed_ids is None else [int(x) for x in removed_ids],
    })

branch = completed.copy()
record_stage("input", len(branch), len(branch))

before = len(branch)
branch = branch.loc[branch["SMILES"].notna()].copy()
record_stage("missing_SMILES", before, len(branch), completed.loc[completed["SMILES"].isna(), "DILIST_ID"])

before = len(branch)
dot_mask = branch["SMILES"].astype(str).str.contains(".", regex=False)
branch = branch.loc[~dot_mask].copy()
record_stage("dot-containing mixture/salt", before, len(branch), completed.loc[completed["SMILES"].notna() & completed["SMILES"].astype(str).str.contains(".", regex=False), "DILIST_ID"])

before = len(branch)
branch["_mol"] = branch["SMILES"].map(parse_smiles)
parse_fail = branch["_mol"].isna()
parse_fail_ids = branch.loc[parse_fail, "DILIST_ID"]
branch = branch.loc[~parse_fail].copy()
record_stage("RDKit parse failure", before, len(branch), parse_fail_ids)

before = len(branch)
mw_mask = branch["_mol"].map(Descriptors.MolWt) >= 1000
mw_ids = branch.loc[mw_mask, "DILIST_ID"]
branch = branch.loc[~mw_mask].copy()
record_stage("molecular weight >= 1000", before, len(branch), mw_ids)

before = len(branch)
carbon_mask = branch["_mol"].map(lambda mol: any(atom.GetSymbol() == "C" for atom in mol.GetAtoms()))
no_carbon_ids = branch.loc[~carbon_mask, "DILIST_ID"]
branch = branch.loc[carbon_mask].copy()
record_stage("no carbon", before, len(branch), no_carbon_ids)

before = len(branch)
metal_mask = branch["_mol"].map(lambda mol: bool({atom.GetSymbol() for atom in mol.GetAtoms()} & METALS))
metal_ids = branch.loc[metal_mask, "DILIST_ID"]
branch = branch.loc[~metal_mask].copy()
record_stage("metal present", before, len(branch), metal_ids)

filter_audit = pd.DataFrame([
    {k: v for k, v in item.items() if k != "removed_ids"} for item in stages
])
display(filter_audit)
print("Final current reconstructed Notebook 1 filtering count:", len(branch))
print("Current branch RDKit parse-failure IDs:", parse_fail_ids.astype(int).tolist())
print("Current branch no-carbon IDs:", no_carbon_ids.astype(int).tolist())
print("Current branch metal exclusions:", int(len(metal_ids)))
print("Current completed-SMILES missing count:", int(completed["SMILES"].isna().sum()))
print("Current completed-SMILES dot-containing count after missing removal:", int(completed.loc[completed["SMILES"].notna(), "SMILES"].astype(str).str.contains(".", regex=False).sum()))

assert len(branch) == 1085
assert int(completed["SMILES"].isna().sum()) == 87
assert int(len(parse_fail_ids)) == 1 and int(parse_fail_ids.iloc[0]) == 397
assert int(len(mw_ids)) == 36
assert int(len(no_carbon_ids)) == 2
assert int(len(metal_ids)) == 0


[23:43:38] SMILES Parse Error: syntax error while parsing: [Fe](|[N]=O)(|[C]#N)(|[C]#N)(|[C]#N)(|[C]#N)|[C]#N
[23:43:38] SMILES Parse Error: check for mistakes around position 6:
[23:43:38] [Fe](|[N]=O)(|[C]#N)(|[C]#N)(|[C]#N)(|[C]
[23:43:38] ~~~~~^
[23:43:38] SMILES Parse Error: Failed parsing SMILES '[Fe](|[N]=O)(|[C]#N)(|[C]#N)(|[C]#N)(|[C]#N)|[C]#N' for input: '[Fe](|[N]=O)(|[C]#N)(|[C]#N)(|[C]#N)(|[C]#N)|[C]#N'


,stage,start_rows,removed,end_rows
0,input,1279,0,1279
1,missing_SMILES,1279,87,1192
2,dot-containing mixture/salt,1192,68,1124
3,RDKit parse failure,1124,1,1123
4,molecular weight >= 1000,1123,36,1087
5,no carbon,1087,2,1085
6,metal present,1085,0,1085


Final current reconstructed Notebook 1 filtering count: 1085
Current branch RDKit parse-failure IDs: [397]
Current branch no-carbon IDs: [191, 617]
Current branch metal exclusions: 0
Current completed-SMILES missing count: 87
Current completed-SMILES dot-containing count after missing removal: 68


### Historical versus reconstructed filtering counts

| Branch | Reported rows | Evidence status |
|---|---:|---|
| Historical Notebook 1 filter | 1,095 | Saved count only; historical filtered rows and complete exclusion IDs are unavailable. |
| Current reconstructed Notebook 1 filter | 1,085 | Recomputed from the recorded 1,279-row completed-SMILES file and original filter conditions. |

**The historical 1,095-row membership cannot be verified from the available evidence; the cause of the discrepancy is not established.** The 1,085-row reconstructed filtering branch is not asserted to be the correct historical membership and is not the accepted modeling cohort.

## 7. Notebook 2 reconstruction and manual mapping record

Original Notebook 2 code supplied six SMILES values for IDs 330, 609, 767, 968, 976, and 998. The values below are transcribed from that notebook solely to verify its recorded transformation. They are labeled `manual_source_unverified`; no external provenance is inferred. Notebook 2 then dropped rows with missing SMILES, resulting in the accepted 1,198-row file. Structure parsing subsequently identifies ID 397 as invalid for structure-based modeling.

In [5]:
# Literal values transcribed from original Notebook 2 code.
# Provenance status for every value: manual_source_unverified.
manual_smiles = {
    330: "CC(=O)N[C@@H]1[C@H]([C@H]([C@H](O[C@H]1O)CO)OS(=O)(=O)[O-])O[C@H]2[C@@H]([C@H]([C@@H]([C@@H](O2)C(=O)[O-])O)O)O",
    609: "CC1([C@@H](N[C@H](S1)C(C(=O)NCCCC[C@H](C(=O)O)N)NC(=O)CC2=CC=CC=C2)C(=O)O)C.C(CCN)C[C@H](C(=O)O)N",
    767: "CN1C(NC2=CC(=C(C=C2S1(=O)=O)S(=O)(=O)N)Cl)CSCC(F)(F)F",
    968: "CN(C)CCCC1C2=CC=CC=C2NC3=C1C=C(C=C3)Cl",
    976: "CC(=O)OC1=CC=C(C=C1)C2(C3=CC=CC=C3NC2=O)C4=CC=C(C=C4)OC(=O)C",
    998: "CC(CC1=CC2=C(C=C1)OCO2)NC",
}

manual_records = []
for dilist_id, smiles_value in manual_smiles.items():
    row_mask = completed["DILIST_ID"].eq(dilist_id)
    if int(row_mask.sum()) != 1:
        raise ValueError(f"Expected exactly one source row for DILIST_ID {dilist_id}.")
    was_missing = bool(completed.loc[row_mask, "SMILES"].isna().iloc[0])
    manual_records.append({
        "DILIST_ID": dilist_id,
        "missing_before_replacement": was_missing,
        "manual_source_status": "manual_source_unverified",
    })

display(pd.DataFrame(manual_records))
print("Manual mapping count:", len(manual_smiles))
print("Missing SMILES before mappings:", int(completed["SMILES"].isna().sum()))

candidate_source = completed.copy()
for dilist_id, smiles_value in manual_smiles.items():
    candidate_source.loc[candidate_source["DILIST_ID"].eq(dilist_id), "SMILES"] = smiles_value

remaining_missing = int(candidate_source["SMILES"].isna().sum())
candidate = candidate_source.dropna(subset=["SMILES"]).reset_index(drop=True)
print("Remaining missing SMILES after six recorded assignments:", remaining_missing)
print("In-memory candidate row count after dropping missing SMILES:", len(candidate))
print("Candidate is in memory only; no dataset file is written.")


,DILIST_ID,missing_before_replacement,manual_source_status
0,330,True,manual_source_unverified
1,609,True,manual_source_unverified
2,767,True,manual_source_unverified
3,968,True,manual_source_unverified
4,976,True,manual_source_unverified
5,998,True,manual_source_unverified


Manual mapping count: 6
Missing SMILES before mappings: 87
Remaining missing SMILES after six recorded assignments: 81
In-memory candidate row count after dropping missing SMILES: 1198
Candidate is in memory only; no dataset file is written.


## 8. Non-destructive candidate versus frozen-file verification

The candidate applies only the six recorded Notebook 2 assignments to the existing completed-SMILES table, then drops missing SMILES as Notebook 2 did. The comparisons below check row count, ordered IDs, SMILES, target, all columns, and candidate serialization hash against the frozen accepted file. A mismatch marks this branch as failed; it does not trigger repair or a write.

In [6]:
comparison_fields = ["DILIST_ID", "SMILES", TARGET]
field_mismatches = {
    column: int((candidate[column].astype("string") != accepted[column].astype("string")).sum())
    for column in comparison_fields
} if len(candidate) == len(accepted) else {column: None for column in comparison_fields}
all_column_mismatches = {}
if len(candidate) == len(accepted) and list(candidate.columns) == list(accepted.columns):
    for column in candidate.columns:
        all_column_mismatches[column] = int(
            (candidate[column].astype("string") != accepted[column].astype("string")).sum()
        )

candidate_hash = hashlib.sha256(candidate.to_csv(index=False).encode("utf-8")).hexdigest()
ids_equal = (
    len(candidate) == len(accepted)
    and candidate["DILIST_ID"].reset_index(drop=True).equals(
        accepted["DILIST_ID"].reset_index(drop=True)
    )
)
row_count_equal = len(candidate) == len(accepted)
schema_equal = list(candidate.columns) == list(accepted.columns)
field_values_equal = row_count_equal and schema_equal and all(v == 0 for v in all_column_mismatches.values())
hash_equal = candidate_hash == accepted_sha256
verification_passed = row_count_equal and ids_equal and schema_equal and field_values_equal and hash_equal

verification = pd.DataFrame([
    {"check": "row count", "candidate": len(candidate), "frozen accepted": len(accepted), "match": row_count_equal},
    {"check": "ordered DILIST_ID", "candidate": "compared", "frozen accepted": "compared", "match": ids_equal},
    {"check": "schema", "candidate": candidate.shape[1], "frozen accepted": accepted.shape[1], "match": schema_equal},
    {"check": "all column values", "candidate": "compared", "frozen accepted": "compared", "match": field_values_equal},
    {"check": "SHA-256", "candidate": candidate_hash, "frozen accepted": accepted_sha256, "match": hash_equal},
])
display(verification)
print("Field-level mismatch counts:", all_column_mismatches)
print("Candidate-vs-frozen verification:", "PASS" if verification_passed else "MISMATCH ? verification branch stopped; no repair performed")

# No write is performed. If a comparison differs, the notebook records the result and
# does not attempt to modify either file or force cohort agreement.


,check,candidate,frozen accepted,match
0,row count,1198,1198,True
1,ordered DILIST_ID,compared,compared,True
2,schema,5,5,True
3,all column values,compared,compared,True
4,SHA-256,4a3c37c704d02b6d0bb80e0727295ca8ba2fd988c4f687...,4a3c37c704d02b6d0bb80e0727295ca8ba2fd988c4f687...,True


Field-level mismatch counts: {'DILIST_ID': 0, 'CompoundName': 0, 'DILIst Classification ': 0, 'Routs of Administration ': 0, 'SMILES': 0}
Candidate-vs-frozen verification: PASS


## 9. Final accepted cohort summary

This summary distinguishes the accepted current modeling cohort from the separate historical Notebook 1 filter branch and the unavailable historical 1,202-row cohort.

In [7]:
cohort_summary = pd.DataFrame([
    {
        "cohort": "Current accepted DILIst_Final.csv (before structure filtering)",
        "rows": len(accepted),
        "class_0": int(accepted_class_counts.get(0, 0)),
        "class_1": int(accepted_class_counts.get(1, 0)),
        "valid_structures": int(valid_mask.sum()),
        "invalid_structure_ids": ", ".join(map(str, invalid_rows["DILIST_ID"].tolist())),
        "sha256": accepted_sha256,
        "status": "accepted current cohort",
    },
    {
        "cohort": "Current reconstructed Notebook 1 filtering branch",
        "rows": len(branch),
        "class_0": None,
        "class_1": None,
        "valid_structures": None,
        "invalid_structure_ids": "397 excluded during filtering",
        "sha256": "not saved as a dataset",
        "status": "reconstructed branch; not historical membership",
    },
    {
        "cohort": "Historical Notebook 1 filtering reference",
        "rows": 1095,
        "class_0": None,
        "class_1": None,
        "valid_structures": None,
        "invalid_structure_ids": "historical membership unavailable",
        "sha256": "unavailable",
        "status": "historical count only; cause not established",
    },
    {
        "cohort": "Historical Notebook 2 reference",
        "rows": 1202,
        "class_0": None,
        "class_1": None,
        "valid_structures": 1201,
        "invalid_structure_ids": "historical row-level file unavailable",
        "sha256": "unavailable",
        "status": "reference only; not identical to current cohort",
    },
])
display(cohort_summary)

print("Positive class semantics documented in the source audit: 1 = DILI Positive; 0 = DILI Negative.")
print("Historical 1,095-row membership cannot be verified; the cause of the discrepancy is not established.")
print("The historical 1,202-row dataset cannot be considered identical to the current 1,198-row cohort without row-level evidence.")


,cohort,rows,class_0,class_1,valid_structures,invalid_structure_ids,sha256,status
0,Current accepted DILIst_Final.csv (before stru...,1198,463.0,735.0,1197.0,397,4a3c37c704d02b6d0bb80e0727295ca8ba2fd988c4f687...,accepted current cohort
1,Current reconstructed Notebook 1 filtering branch,1085,NaN,NaN,NaN,397 excluded during filtering,not saved as a dataset,reconstructed branch; not historical membership
2,Historical Notebook 1 filtering reference,1095,NaN,NaN,NaN,historical membership unavailable,unavailable,historical count only; cause not established
3,Historical Notebook 2 reference,1202,NaN,NaN,1201.0,historical row-level file unavailable,unavailable,reference only; not identical to current cohort


Positive class semantics documented in the source audit: 1 = DILI Positive; 0 = DILI Negative.
Historical 1,095-row membership cannot be verified; the cause of the discrepancy is not established.
The historical 1,202-row dataset cannot be considered identical to the current 1,198-row cohort without row-level evidence.


## 10. Limitations and provenance warnings

- The historical Notebook 1 output retained a count of 1,095 but not the complete filtered rows or exclusion IDs. The current reconstructed filter yields 1,085; **the historical 1,095-row membership cannot be verified from the available evidence; the cause of the discrepancy is not established.**
- The historical Notebook 2 input with 1,202 rows is unavailable for row-level comparison. The accepted 1,198-row cohort is not claimed to be identical to it.
- Six manual SMILES assignments are recorded in original Notebook 2, but their source records are unavailable. They remain `manual_source_unverified` and are not described as externally verified.
- DILIST_ID 397 is present in the accepted CSV and contributes to its pre-structure-filter row/class counts. RDKit parsing fails for this record, so it is excluded only from structure-based modeling, leaving 1,197 valid structures.
- This notebook makes no live PubChem/NCI Cactus requests and performs no model training.
- All verification is read-only with respect to the accepted CSV and original notebooks.